# TinyChirp CNN-Time TensorFlow

Train a 1D CNN on raw audio waveforms (no spectrogram), export an int8 TFLite model, and write a Rust `audio_sample.rs` file.

This mirrors `building_tensorflow/tinychirp_cnn.ipynb` but replaces the 2D mel CNN with a 1D time CNN similar to `StreamingCNNArch` from the TinyChirp `CNN_Time` model.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from typing import TYPE_CHECKING

from utils import (
    TARGET_AUDIO_LEN_TIME,
    get_paths,
    configure_tf_runtime,
    set_global_seed,
)

if TYPE_CHECKING:
    import keras
else:
    from tensorflow import keras

configure_tf_runtime()
set_global_seed()

MODEL_STEM = "cnn_time_tf"
paths = get_paths(MODEL_STEM)
OUT_TFLITE = paths.out_tflite
OUT_AUDIO_RS = paths.out_audio_rs
BATCH_SIZE = 32

In [3]:
from utils import make_time_datasets

train_ds, val_ds, test_ds, label_names = make_time_datasets()
num_labels = len(label_names)
print("Classes:", label_names)

Found 11292 files belonging to 2 classes.
Found 1380 files belonging to 2 classes.
Found 1393 files belonging to 2 classes.
Classes: ['non_target' 'target']


In [4]:
model = keras.Sequential([
    keras.layers.Input(shape=(TARGET_AUDIO_LEN_TIME, 1)),
    keras.layers.Conv1D(4, 3, activation="relu"),
    keras.layers.AveragePooling1D(pool_size=2, strides=2),
    keras.layers.Conv1D(8, 3, activation="relu"),
    # GlobalAveragePooling1D maps to MEAN in TFLite which microflow doesn't support.
    # AveragePooling1D maps to AVERAGE_POOL_2D — cover the full time dimension instead.
    keras.layers.AveragePooling1D(pool_size=23933, padding="valid", name="final_pool"),
    keras.layers.Flatten(name="flatten"),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(num_labels),
])


model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 47870, 4)       │            16 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling1d               │ (None, 23935, 4)       │             0 │
│ (AveragePooling1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 23933, 8)       │           104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ final_pool (AveragePooling1D)   │ (None, 1, 8)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 826 (3.23 KB)

 Trainable params: 826 (3.23 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
from utils import init_wandb, get_callbacks, finish_wandb

init_wandb(MODEL_STEM, config={
    "conv1_filters": 4,
    "conv2_filters": 8,
    "kernel_size": 3,
    "dense_units": 64,
})

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    validation_steps=50,
    callbacks=get_callbacks(10,5, BATCH_SIZE)
)
finish_wandb()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/nathan/.netrc.
wandb: Currently logged in as: nathan-duboisset (nathan-duboisset-cole-polytechnique) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/100


I0000 00:00:1776684416.009577   18797 service.cc:145] XLA service 0x7ba558005670 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776684416.009625   18797 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9


  3/353 ━━━━━━━━━━━━━━━━━━━━ 20s 60ms/step - accuracy: 0.6441 - loss: 0.6907 

I0000 00:00:1776684426.101181   18797 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


353/353 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.6690 - loss: 0.6281

/home/nathan/Documents/tiny-chirp-microflow/building_tensorflow/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


353/353 ━━━━━━━━━━━━━━━━━━━━ 48s 104ms/step - accuracy: 0.6776 - loss: 0.5817 - val_accuracy: 0.7370 - val_loss: 0.4745
Epoch 2/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 29s 81ms/step - accuracy: 0.7666 - loss: 0.4508 - val_accuracy: 0.8688 - val_loss: 0.3988
Epoch 3/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 27s 76ms/step - accuracy: 0.8039 - loss: 0.3869 - val_accuracy: 0.9051 - val_loss: 0.3262
Epoch 4/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 30s 84ms/step - accuracy: 0.8603 - loss: 0.3257 - val_accuracy: 0.9014 - val_loss: 0.2752
Epoch 5/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 32s 89ms/step - accuracy: 0.8827 - loss: 0.2920 - val_accuracy: 0.9225 - val_loss: 0.2431
Epoch 6/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 30s 84ms/step - accuracy: 0.8946 - loss: 0.2760 - val_accuracy: 0.9181 - val_loss: 0.2290
Epoch 7/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 31s 87ms/step - accuracy: 0.9013 - loss: 0.2637 - val_accuracy: 0.9101 - val_loss: 0.2190
Epoch 8/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 29s 81ms/step - accuracy: 0.9093 - loss: 0.2519 - val

batch/accuracy,▁▁▁▄▆▇▇▇▇▇▇▇██▇█████████████████████████
batch/batch_step,▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇█████
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,█▆▇▆▅▃▃▃▃▃▂▂▂▂▂▂▂▂▂▄▁▂▁▂▁▁▁▂▂▂▁▁▁▁▁▁▁▂▁▂
epoch/accuracy,▁▃▄▆▇▇▇▇▇▇▇▇▇███████████████████████████
epoch/epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▄▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▅▆▇███▇███████▇██▇████▇█████████▇██████
epoch/val_loss,█▇▃▃▃▂▂▂▂▂▁▂▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▂▁▁▁▁▂▁▁
+6,...


In [6]:
from utils import (
    build_representative_batches,
    export_keras_model_to_int8_tflite,
)

export_inputs = keras.Input(shape=(TARGET_AUDIO_LEN_TIME, 1), name="audio_waveform")
export_outputs = model(export_inputs)
export_model = keras.Model(export_inputs, export_outputs, name="cnn_time_microflow")

val_specs = build_representative_batches(test_ds, take=100)

export_keras_model_to_int8_tflite(export_model, val_specs, OUT_TFLITE)

Saved artifact at '/tmp/tmpx4nqmpcw'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 47872), dtype=tf.float32, name=None)
Output Type:
  TensorSpec(shape=(1, 2), dtype=tf.float32, name=None)
Captures:
  135954576240736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135954576241440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135954576241616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135954576242496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135954576241088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135954576241968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135954576242320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135954576242848: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1776686760.875755   18600 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1776686760.876239   18600 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


In [7]:
from utils import evaluate_tflite_model

train_m, test_m, avg_ms = evaluate_tflite_model(OUT_TFLITE, MODEL_STEM, train_ds, test_ds)
print(f"Avg inference: {avg_ms:.3f} ms")

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


ValueError: Cannot set tensor: Dimension mismatch. Got 3 but expected 2 for input 0.